# Лабораторная работа №4. Генерация текста с механизмом внимания (RuGPT)

**Цель:** Заставить языковую модель сгенерировать заданные слова в указанном порядке, не используя их в промпте и не применяя принудительную вставку.
Целевые слова: «индейка», «шампиньоны» .

**Использованный промпт:**

Классификация продуктов для праздничного стола.

- Мясо птицы по размеру:
- Мелкое — курица.
- Среднее — утка.
- Крупное —

In [2]:
import re
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer, set_seed

MODEL_NAME = "ai-forever/rugpt3large_based_on_gpt2"
TARGET_WORDS = ["индейка", "шампиньоны"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

PROMPTS = [
    "Классификация продуктов для праздничного стола.\n"
    "Мясо птицы по размеру:\n"
    "Мелкое — курица.\n"
    "Среднее — утка.\n"
    "Крупное — ",

    "Виды мяса птицы:\n"
    "Самое распространённое — курица.\n"
    "Более крупное и диетическое — ",

    "Продукты для запекания:\n"
    "Мясо птицы — ",

    "Меню ресторана:\n"
    "Блюда из крупной птицы — ",

    "Кулинарный словарь.\n"
    "Крупная домашняя птица с белым мясом — ",
]

TEMPERATURES = [0.7, 0.8, 0.9]
SEEDS = list(range(100, 150))  # 50 seed

def check_prompt(text):
    words = re.findall(r"\w+", text.casefold())
    if any(w.startswith(("индейк", "шампиньон")) for w in words):
        raise ValueError("Промпт содержит запрещённую словоформу!")

def check_result(text):
    words = re.findall(r"\w+", text.casefold())
    first_seen = False
    for word in words:
        if word == TARGET_WORDS[0]:
            first_seen = True
        elif word == TARGET_WORDS[1] and first_seen:
            return True
    return False

def generate(model, tok, prompt, temperature, seed, max_new_tokens=200):
    check_prompt(prompt)
    set_seed(seed)
    inputs = tok(prompt, return_tensors="pt").to(DEVICE)
    prompt_length = inputs["input_ids"].shape[1]
    with torch.inference_mode():
        output = model.generate(
            **inputs, do_sample=True, temperature=temperature,
            top_k=50, top_p=0.95, max_new_tokens=max_new_tokens,
            repetition_penalty=1.0, no_repeat_ngram_size=0,
        )
    return tok.decode(output[0, prompt_length:], skip_special_tokens=True)

def main():
    print(f"Устройство: {DEVICE}")
    print("Загрузка модели...")
    tok = GPT2Tokenizer.from_pretrained(MODEL_NAME)
    model = GPT2LMHeadModel.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    ).to(DEVICE)
    model.eval()
    print(" Модель загружена.\n")

    total = len(PROMPTS) * len(TEMPERATURES) * len(SEEDS)
    print(f"Запланировано запусков: {total}\n")

    attempt = 0
    for prompt_idx, prompt in enumerate(PROMPTS, 1):
        for temp in TEMPERATURES:
            for seed in SEEDS:
                attempt += 1
                continuation = generate(model, tok, prompt, temp, seed)
                success = check_result(continuation)
                print(f"[{attempt}/{total}] Промпт #{prompt_idx} | "
                      f"T={temp} | seed={seed} | успех={success}")
                if success:
                    print("\n" + "=" * 60)
                    print("НАШЛИ УСПЕШНЫЙ ЗАПУСК!")
                    print("=" * 60)
                    print(f"\nПромпт:\n{prompt}")
                    print(f"\nПараметры: temperature={temp}, seed={seed}")
                    print(f"\nПродолжение:\n{continuation}")
                    print("=" * 60)
                    return

    print("\n Ни одна конфигурация не сработала.")

if __name__ == "__main__":
    main()

Устройство: cuda
Загрузка модели...


tokenizer_config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.71M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.27M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/574 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

... (5 outputs left out)

[transformers] GPT2LMHeadModel LOAD REPORT from: ai-forever/rugpt3large_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Модель загружена.

Запланировано запусков: 750



[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[1/750] Промпт #1 | T=0.7 | seed=100 | успех=False
[2/750] Промпт #1 | T=0.7 | seed=101 | успех=False
[3/750] Промпт #1 | T=0.7 | seed=102 | успех=False
[4/750] Промпт #1 | T=0.7 | seed=103 | успех=False
[5/750] Промпт #1 | T=0.7 | seed=104 | успех=False
[6/750] Промпт #1 | T=0.7 | seed=105 | успех=False
[7/750] Промпт #1 | T=0.7 | seed=106 | успех=False
[8/750] Промпт #1 | T=0.7 | seed=107 | успех=False
[9/750] Промпт #1 | T=0.7 | seed=108 | успех=False
[10/750] Промпт #1 | T=0.7 | seed=109 | успех=False
[11/750] Промпт #1 | T=0.7 | seed=110 | успех=False
[12/750] Промпт #1 | T=0.7 | seed=111 | успех=False
[13/750] Промпт #1 | T=0.7 | seed=112 | успех=False
[14/750] Промпт #1 | T=0.7 | seed=113 | успех=False
[15/750] Промпт #1 | T=0.7 | seed=114 | успех=False
[16/750] Промпт #1 | T=0.7 | seed=115 | успех=False
[17/750] Промпт #1 | T=0.7 | seed=116 | успех=False
[18/750] Промпт #1 | T=0.7 | seed=117 | успех=False
[19/750] Промпт #1 | T=0.7 | seed=118 | успех=False
[20/750] Промпт #1 | 

**Параметры генерации, при которых достигнут успех:**
- temperature: 0.9
- seed: 127
- top_k: 50
- top_p: 0.95
- max_new_tokens: 200

**Результат:** Модель успешно сгенерировала текст, содержащий слова «индейка» и «шампиньоны» в заданном порядке. Скриншот выполнения кода прилагается.